<a href="https://colab.research.google.com/github/Brahamaulakh/DataStructure/blob/new/itenary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q transformers datasets accelerate peft fastapi uvicorn pyngrok

In [9]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
GPU Available: False


In [10]:
import transformers
import datasets
import accelerate
import peft
import fastapi
import uvicorn
import pyngrok

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Pyngrok:", pyngrok.__version__)

Transformers: 5.15.0
Datasets: 4.0.0
Accelerate: 1.14.0
PEFT: 0.20.0
FastAPI: 0.141.1
Uvicorn: 0.52.3
Pyngrok: 8.1.2


In [11]:
import os

os.makedirs("/content/voucher_model/data", exist_ok=True)
os.makedirs("/content/voucher_model/model", exist_ok=True)

print("Project ready")

Project ready


In [12]:
voucher_text = """
ITINERARY:
12th Aug - Arrived Hong Kong International Airport
15:30 - After Immigration, collect your luggage and proceed to meet the driver at
 Arrival Hall B and walk to Limousine Lounge
 Transfer to City Hotel
17:00 - Post Lantau, proceed to HONG KONG NIGHT TOUR (5-HOUR TOUR)
 Visit – Victoria Peak with 1-way Peak Tram ride
 1-way Star Ferry ride Over Victoria Harbour
 (NO Physical ticket! Driver will pay or hand over cash for the same on spot)
 Old clock tower & Symphony of light Show (SOL) at TST Promenade
 Post tour drop back to the hotel
13th Aug - Hong Kong (B)
 Breakfast at Hotel
09:00 - Morning pick up from hotel lobby & transfer to DISNEYLAND THEME PARK
 Enjoy a full day at Disneyland theme park (1-day Entrance Pass)
21:30 - Post fireworks. Pick-up from Car Park No.4 & Transfer back to Hotel
14th Aug – Hong Kong (B)
 Breakfast at Hotel
12:00 - Pick-up from the hotel Lobby for LANTAU ISLAND TOUR
 Enjoy 01 Way Cable Car ride up to the Ngong Ping Station – Standard Cabin
 Visit the Ngong Ping Village, Po Lin Monastery, Grand Buddha Statue
 Enjoy 01 Way Cable Car ride down to Tung Chung Station – Standard Cabin
 Enjoy Shopping at City Gate Shopping Mall – Duty Free & Branded Shopping
18:00 - Post tour, transfer back to Hotel.

15th Aug - Hong Kong (B)
 Breakfast at Hotel
10:00 - Morning pick up from hotel lobby & transfer to OCEAN PARK THEME PARK
 Enjoy a full day at Ocean Park
19:30 - Evening pick-up from drop off area & transfer back to the Hotel.
16th Aug - Hong Kong / Shenzhen (B)
 Check out from Hotel
Breakfast at Hotel
08:00 - Morning pick up from hotel lobby & transfer to HungHom Train Sation
 Train transit from HungHom to Louhu Border – first class ticket (w/ HK Guide)
 Cross Immigration – China visa provided
 Meet Shenzhen Guide and transfer to Hotel and check in
17th Aug - Shenzhen (B,L)
 Breakfast at Hotel
10:00 - Morning pick up from Hotel lobby for Shenzhen City tour w/ Indian Lunch - Private
 Visit : Compulsory Jade shop, Lotus Park, Windows of the World
 Lunch at Indian Restaurant
 Shopping at Louhu Shopping Mall
 Post tour, transfer to Hotel and check in
18th Aug - Shenzhen (B)
 Breakfast at Hotel
 FREE DAY at own leisure
19th Aug – Shenzhen / Macau (B)
 Breakfast at Hotel
 Check out from Hotel
10:00 - Pick up from hotel lobby for transfer to Shekou ferry pier
 Cross Immigration
12:00 - Transit by ferry to Taipa ferry Terminal – economy class
13:00 - Cross immigration
13:30 - Arrived Macau, meet driver and transfer to Hotel – Private

20th Aug – Macau (B)
 Breakfast at Hotel
13:00 - Pick up at Hotel Lobby for MACAU CITY TOUR – (4-HR TOUR)
 Visit – Fishermen’s Wharf, Ruins of St. Paul, Senado Square, Venetian Hotel
 Londoner Hotel, Parisian Garden Eiffel Tower (outside for photo stop)
 Water/Fountain Show & Sky Cab at Wynn Palace.
 Macau Tower (outside for photo stop)
 Post Tour transfer back to hotel
21st Aug – Macau / Hong Kong (B)
 Breakfast at Hotel
 Check out from Hotel
07:30 - Pick up from Hotel lobby HZMB Macau Port
 Please arrive at the check-in counter at least 90 minutes before the bus departure time.
 Cross Macau Immigration
09:30 - DIRECT Airport Bus Shuttle from Macau to HK Airport
 Fly back to the destination
"""

In [13]:
with open("/content/voucher_model/data/voucher_001.txt", "w", encoding="utf-8") as f:
    f.write(voucher_text)

print("Voucher 001 saved.")

Voucher 001 saved.


In [14]:
import json

voucher_001 = {
    "voucher_id": "voucher_001",

    "trip": {
        "arrival_date": "2026-08-12",
        "departure_date": "2026-08-21",
        "destinations": [
            "Hong Kong",
            "Shenzhen",
            "Macau"
        ]
    },

    "daily_itinerary": [
        {
            "date": "2026-08-12",
            "location": "Hong Kong",
            "meals": [],
            "activities": [
                {
                    "time": "15:30",
                    "type": "airport_transfer",
                    "description": "Arrival at Hong Kong International Airport, immigration, luggage collection and transfer to City Hotel"
                },
                {
                    "time": "17:00",
                    "type": "sightseeing",
                    "name": "Hong Kong Night Tour",
                    "duration": "5 hours",
                    "activities": [
                        "Victoria Peak",
                        "1-way Peak Tram",
                        "1-way Star Ferry",
                        "Victoria Harbour",
                        "Old Clock Tower",
                        "Symphony of Lights",
                        "TST Promenade"
                    ]
                }
            ]
        },

        {
            "date": "2026-08-13",
            "location": "Hong Kong",
            "meals": ["Breakfast"],
            "activities": [
                {
                    "time": "09:00",
                    "type": "theme_park",
                    "name": "Hong Kong Disneyland",
                    "duration": "Full day",
                    "inclusions": ["1-day entrance pass"]
                },
                {
                    "time": "21:30",
                    "type": "transfer",
                    "description": "Pickup from Car Park No. 4 and transfer to hotel"
                }
            ]
        },

        {
            "date": "2026-08-14",
            "location": "Hong Kong",
            "meals": ["Breakfast"],
            "activities": [
                {
                    "time": "12:00",
                    "type": "tour",
                    "name": "Lantau Island Tour",
                    "activities": [
                        "Ngong Ping Cable Car - standard cabin",
                        "Ngong Ping Village",
                        "Po Lin Monastery",
                        "Grand Buddha Statue",
                        "Ngong Ping Cable Car return",
                        "Citygate Shopping Mall"
                    ]
                },
                {
                    "time": "18:00",
                    "type": "transfer",
                    "description": "Transfer back to hotel"
                }
            ]
        },

        {
            "date": "2026-08-15",
            "location": "Hong Kong",
            "meals": ["Breakfast"],
            "activities": [
                {
                    "time": "10:00",
                    "type": "theme_park",
                    "name": "Ocean Park",
                    "duration": "Full day"
                },
                {
                    "time": "19:30",
                    "type": "transfer",
                    "description": "Pickup and transfer back to hotel"
                }
            ]
        },

        {
            "date": "2026-08-16",
            "location": "Hong Kong → Shenzhen",
            "meals": ["Breakfast"],
            "hotel_change": True,
            "activities": [
                {
                    "time": "08:00",
                    "type": "intercity_transfer",
                    "transport": "Train",
                    "route": "Hung Hom → Lo Wu Border",
                    "class": "First Class",
                    "guide": True
                },
                {
                    "type": "immigration",
                    "description": "Cross immigration with China visa provided"
                },
                {
                    "type": "hotel_transfer",
                    "description": "Meet Shenzhen guide and transfer to hotel"
                }
            ]
        },

        {
            "date": "2026-08-17",
            "location": "Shenzhen",
            "meals": ["Breakfast", "Lunch"],
            "activities": [
                {
                    "time": "10:00",
                    "type": "city_tour",
                    "name": "Shenzhen City Tour",
                    "private": True,
                    "activities": [
                        "Compulsory Jade Shop",
                        "Lotus Park",
                        "Windows of the World",
                        "Indian Lunch",
                        "Louhu Shopping Mall"
                    ]
                },
                {
                    "type": "transfer",
                    "description": "Transfer back to hotel"
                }
            ]
        },

        {
            "date": "2026-08-18",
            "location": "Shenzhen",
            "meals": ["Breakfast"],
            "activities": [
                {
                    "type": "free_day",
                    "description": "Free day at own leisure"
                }
            ]
        },

        {
            "date": "2026-08-19",
            "location": "Shenzhen → Macau",
            "meals": ["Breakfast"],
            "hotel_change": True,
            "activities": [
                {
                    "time": "10:00",
                    "type": "transfer",
                    "description": "Transfer from hotel to Shekou Ferry Pier"
                },
                {
                    "type": "immigration",
                    "description": "Cross immigration"
                },
                {
                    "time": "12:00",
                    "type": "intercity_transfer",
                    "transport": "Ferry",
                    "route": "Shekou → Taipa Ferry Terminal",
                    "class": "Economy"
                },
                {
                    "time": "13:00",
                    "type": "immigration",
                    "description": "Macau immigration"
                },
                {
                    "time": "13:30",
                    "type": "hotel_transfer",
                    "description": "Meet driver and transfer to Macau hotel"
                }
            ]
        },

        {
            "date": "2026-08-20",
            "location": "Macau",
            "meals": ["Breakfast"],
            "activities": [
                {
                    "time": "13:00",
                    "type": "city_tour",
                    "name": "Macau City Tour",
                    "duration": "4 hours",
                    "activities": [
                        "Fishermen's Wharf",
                        "Ruins of St. Paul",
                        "Senado Square",
                        "Venetian Hotel",
                        "Londoner Hotel",
                        "Parisian Garden Eiffel Tower",
                        "Wynn Palace Water/Fountain Show",
                        "Sky Cab at Wynn Palace",
                        "Macau Tower"
                    ]
                },
                {
                    "type": "transfer",
                    "description": "Transfer back to hotel"
                }
            ]
        },

        {
            "date": "2026-08-21",
            "location": "Macau → Hong Kong Airport",
            "meals": ["Breakfast"],
            "hotel_change": True,
            "activities": [
                {
                    "time": "07:30",
                    "type": "airport_transfer",
                    "description": "Transfer from hotel to HZMB Macau Port"
                },
                {
                    "type": "immigration",
                    "description": "Cross Macau immigration"
                },
                {
                    "time": "09:30",
                    "type": "airport_transfer",
                    "transport": "Direct Airport Bus Shuttle",
                    "route": "Macau → Hong Kong Airport"
                },
                {
                    "type": "flight",
                    "description": "Fly back to destination"
                }
            ]
        }
    ],

    "constraints": {
        "airport_transfer_wait_minutes": 90,
        "private_transfer_wait_minutes": 15,
        "required_documents": [
            "Printed PAR for Indian passport holders",
            "Original passport"
        ],
        "weather_disruption": [
            "Typhoon Signal No. 8 or above",
            "Black Rainstorm warning"
        ]
    }
}

with open(
    "/content/voucher_model/data/voucher_001.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(voucher_001, f, indent=2, ensure_ascii=False)

print("voucher_001.json created successfully.")

voucher_001.json created successfully.


In [15]:
import json

with open(
    "/content/voucher_model/data/voucher_001.json",
    "r",
    encoding="utf-8"
) as f:
    voucher = json.load(f)

training_example = {
    "input": json.dumps(voucher, ensure_ascii=False),
    "output": json.dumps(
        voucher["daily_itinerary"],
        ensure_ascii=False
    )
}

with open(
    "/content/voucher_model/data/training.jsonl",
    "w",
    encoding="utf-8"
) as f:
    f.write(json.dumps(training_example, ensure_ascii=False) + "\n")

print("training.jsonl created successfully.")

training.jsonl created successfully.


In [16]:
with open(
    "/content/voucher_model/data/training.jsonl",
    "r",
    encoding="utf-8"
) as f:
    line = f.readline()

example = json.loads(line)

print("Input characters:", len(example["input"]))
print("Output characters:", len(example["output"]))

Input characters: 4435
Output characters: 3998


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
import json
import torch

# Load voucher
with open(
    "/content/voucher_model/data/voucher_001.json",
    "r",
    encoding="utf-8"
) as f:
    voucher = json.load(f)

voucher_text = json.dumps(voucher, ensure_ascii=False, indent=2)

prompt = f"""
You are an expert travel itinerary planner.

Using the voucher information below, create ONE optimized travel itinerary.

IMPORTANT:
- Keep all fixed transportation timings.
- Keep the arrival and departure dates.
- Do not remove included activities.
- Do not invent transportation that is not present.
- Respect hotel changes.
- Respect travel direction between destinations.
- Respect activity durations.
- Include realistic transfer time.
- Avoid overlapping activities.
- Use exact timings where they are provided.
- For activities without fixed timings, assign realistic timings.
- Clearly show date, location, time, activity and transportation.

Return ONLY the itinerary.

VOUCHER:
{voucher_text}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=3000,
        temperature=0.8,
        top_p=0.9,
        do_sample=True
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

result = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(result)

In [ ]:
import json
import torch

# Load voucher
with open(
    "/content/voucher_model/data/voucher_001.json",
    "r",
    encoding="utf-8"
) as f:
    voucher = json.load(f)

voucher_text = json.dumps(
    voucher,
    ensure_ascii=False,
    indent=2
)

prompt = f"""
You are an expert travel itinerary optimization engine.

Your task is to create EXACTLY 9 DIFFERENT itinerary variations
from the SAME travel voucher.

CRITICAL RULES:

1. DO NOT change the trip dates.
2. DO NOT remove any included activity.
3. DO NOT invent activities.
4. DO NOT change fixed transportation timings.
5. DO NOT change fixed ferry/train/bus timings.
6. DO NOT change airport departure timings.
7. DO NOT create overlapping activities.
8. Every activity must have a realistic start and end time.
9. Include realistic transfer time between activities.
10. Activities in the same city may be rearranged when possible.
11. Activities involving intercity transportation must respect the
    transportation schedule.
12. Hotel changes must remain consistent with the voucher.
13. Maintain all meals specified in the voucher.
14. Free days may remain free.
15. Each variation must be meaningfully different where the
    constraints allow it.
16. Do not create impossible travel.
17. Keep the exact fixed times from the voucher.

FIXED EVENTS MUST NEVER BE MODIFIED.

Your output MUST contain exactly 9 variations.

Use this format:

VARIATION 1
DATE
LOCATION
TIME
ACTIVITY
DURATION
TRANSFER

VARIATION 2
DATE
LOCATION
TIME
ACTIVITY
DURATION
TRANSFER

Continue until VARIATION 9.

VOUCHER:

{voucher_text}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=8000,
        temperature=0.95,
        top_p=0.9,
        do_sample=True
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

result = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(result)

In [ ]:
allowed_activities = [
    "Hong Kong Night Tour",
    "Disneyland Theme Park",
    "Lantau Island Tour",
    "Ocean Park Theme Park",
    "Shenzhen City Tour",
    "Free Day",
    "Macau City Tour"
]

fixed_events = [
    {
        "date": "2026-08-12",
        "time": "15:30",
        "activity": "Hong Kong International Airport arrival and transfer to hotel"
    },
    {
        "date": "2026-08-12",
        "time": "17:00",
        "activity": "Hong Kong Night Tour"
    },
    {
        "date": "2026-08-13",
        "time": "09:00",
        "activity": "Disneyland Theme Park"
    },
    {
        "date": "2026-08-13",
        "time": "21:30",
        "activity": "Disneyland pickup and hotel transfer"
    },
    {
        "date": "2026-08-14",
        "time": "12:00",
        "activity": "Lantau Island Tour"
    },
    {
        "date": "2026-08-14",
        "time": "18:00",
        "activity": "Return to hotel"
    },
    {
        "date": "2026-08-15",
        "time": "10:00",
        "activity": "Ocean Park Theme Park"
    },
    {
        "date": "2026-08-15",
        "time": "19:30",
        "activity": "Return to hotel"
    },
    {
        "date": "2026-08-16",
        "time": "08:00",
        "activity": "Hong Kong to Shenzhen transfer"
    },
    {
        "date": "2026-08-17",
        "time": "10:00",
        "activity": "Shenzhen City Tour"
    },
    {
        "date": "2026-08-18",
        "time": None,
        "activity": "Free Day"
    },
    {
        "date": "2026-08-19",
        "time": "10:00",
        "activity": "Transfer to Shekou Ferry Pier"
    },
    {
        "date": "2026-08-19",
        "time": "12:00",
        "activity": "Ferry to Taipa"
    },
    {
        "date": "2026-08-19",
        "time": "13:00",
        "activity": "Macau Immigration"
    },
    {
        "date": "2026-08-19",
        "time": "13:30",
        "activity": "Transfer to Macau Hotel"
    },
    {
        "date": "2026-08-20",
        "time": "13:00",
        "activity": "Macau City Tour"
    },
    {
        "date": "2026-08-21",
        "time": "07:30",
        "activity": "Transfer to HZMB Macau Port"
    },
    {
        "date": "2026-08-21",
        "time": "09:30",
        "activity": "Direct Airport Bus to Hong Kong Airport"
    }
]

print("Allowed activities:", len(allowed_activities))
print("Fixed events:", len(fixed_events))

In [ ]:
import json
import torch

# Load voucher
with open(
    "/content/voucher_model/data/voucher_001.json",
    "r",
    encoding="utf-8"
) as f:
    voucher = json.load(f)

voucher_text = json.dumps(
    voucher,
    ensure_ascii=False,
    indent=2
)

prompt = f"""
You are an expert travel itinerary optimization engine.

Your task is to create EXACTLY 9 DIFFERENT itinerary variations
from the SAME travel voucher.

CRITICAL RULES:

1. DO NOT change the trip dates.
2. DO NOT remove any included activity.
3. DO NOT invent activities.
4. DO NOT change fixed transportation timings.
5. DO NOT change fixed ferry/train/bus timings.
6. DO NOT change airport departure timings.
7. DO NOT create overlapping activities.
8. Every activity must have a realistic start and end time.
9. Include realistic transfer time between activities.
10. Activities in the same city may be rearranged when possible.
11. Activities involving intercity transportation must respect the
    transportation schedule.
12. Hotel changes must remain consistent with the voucher.
13. Maintain all meals specified in the voucher.
14. Free days may remain free.
15. Each variation must be meaningfully different where the
    constraints allow it.
16. Do not create impossible travel.
17. Keep the exact fixed times from the voucher.

FIXED EVENTS MUST NEVER BE MODIFIED.

Your output MUST contain exactly 9 variations.

Use this format:

VARIATION 1
DATE
LOCATION
TIME
ACTIVITY
DURATION
TRANSFER

VARIATION 2
DATE
LOCATION
TIME
ACTIVITY
DURATION
TRANSFER

Continue until VARIATION 9.

VOUCHER:

{voucher_text}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=8000,
        temperature=0.95,
        top_p=0.9,
        do_sample=True
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

result = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(result)

In [ ]:
from itertools import permutations

cities = [
    "Hong Kong",
    "Shenzhen",
    "Macau"
]

city_sequences = list(permutations(cities))

for i, sequence in enumerate(city_sequences, 1):
    print(f"Variation {i}: {' → '.join(sequence)}")

In [ ]:
travel_constraints = {
    "arrival": {
        "date": "2026-08-12",
        "city": "Hong Kong",
        "time": "15:30"
    },

    "departure": {
        "date": "2026-08-21",
        "city": "Hong Kong Airport",
        "time": None
    },

    "required_transfers": [
        {
            "date": "2026-08-16",
            "from": "Hong Kong",
            "to": "Shenzhen",
            "time": "08:00",
            "transport": "Train"
        },
        {
            "date": "2026-08-19",
            "from": "Shenzhen",
            "to": "Macau",
            "time": "12:00",
            "transport": "Ferry"
        },
        {
            "date": "2026-08-21",
            "from": "Macau",
            "to": "Hong Kong Airport",
            "time": "09:30",
            "transport": "Airport Bus"
        }
    ]
}

print("Travel constraints loaded.")

In [ ]:
def is_sequence_valid(sequence, constraints):
    """
    Check whether a city sequence respects the fixed
    intercity transportation in the voucher.
    """

    sequence = list(sequence)

    arrival_city = constraints["arrival"]["city"]

    # Trip must start in arrival city
    if sequence[0] != arrival_city:
        return False, f"Trip starts in {sequence[0]}, but arrival is in {arrival_city}"

    # Fixed transfers must follow the city order
    required_route = [
        constraints["required_transfers"][0]["from"],
        constraints["required_transfers"][0]["to"],
        constraints["required_transfers"][1]["to"]
    ]

    # Required route:
    # Hong Kong → Shenzhen → Macau

    if sequence != required_route:
        return False, (
            f"Voucher fixes the route as "
            f"{' → '.join(required_route)}"
        )

    return True, "Valid"


valid_sequences = []

for i, sequence in enumerate(city_sequences, 1):

    valid, reason = is_sequence_valid(
        sequence,
        travel_constraints
    )

    print(
        f"Variation {i}: {' → '.join(sequence)}"
    )
    print(
        "VALID" if valid else f"INVALID: {reason}"
    )
    print()

    if valid:
        valid_sequences.append(sequence)

In [ ]:
reference_locations = {
    "cities": [
        "Hong Kong",
        "Shenzhen",
        "Macau"
    ],

    "hong_kong": [
        "Hong Kong International Airport",
        "Victoria Peak",
        "Peak Tram",
        "Star Ferry",
        "Victoria Harbour",
        "Old Clock Tower",
        "TST Promenade",
        "Hong Kong Disneyland",
        "Lantau Island",
        "Ngong Ping Village",
        "Po Lin Monastery",
        "Grand Buddha Statue",
        "Citygate Shopping Mall",
        "Ocean Park"
    ],

    "shenzhen": [
        "Hung Hom Station",
        "Lo Wu Border",
        "Lotus Park",
        "Windows of the World",
        "Jade Shop",
        "Louhu Shopping Mall"
    ],

    "macau": [
        "Shekou Ferry Pier",
        "Taipa Ferry Terminal",
        "Fishermen's Wharf",
        "Ruins of St. Paul",
        "Senado Square",
        "Venetian Hotel",
        "Londoner Hotel",
        "Parisian Garden Eiffel Tower",
        "Wynn Palace",
        "Macau Tower",
        "HZMB Macau Port"
    ]
}

print("Cities:", reference_locations["cities"])
print("Hong Kong locations:", len(reference_locations["hong_kong"]))
print("Shenzhen locations:", len(reference_locations["shenzhen"]))
print("Macau locations:", len(reference_locations["macau"]))

In [ ]:
from itertools import permutations

cities = reference_locations["cities"]

city_variations = []

for i, sequence in enumerate(permutations(cities), 1):
    city_variations.append({
        "variation_id": i,
        "city_order": list(sequence),
        "locations": {
            city: reference_locations[city.lower().replace(" ", "_")]
            for city in sequence
        }
    })

for variation in city_variations:
    print(
        f"Variation {variation['variation_id']}: "
        f"{' → '.join(variation['city_order'])}"
    )

In [ ]:
city_blocks = {

    "Hong Kong": {
        "days": 4,
        "activities": [
            {
                "name": "Hong Kong Night Tour",
                "time": "17:00",
                "duration": "5 hours",
                "locations": [
                    "Victoria Peak",
                    "Peak Tram",
                    "Star Ferry",
                    "Victoria Harbour",
                    "Old Clock Tower",
                    "TST Promenade"
                ]
            },
            {
                "name": "Hong Kong Disneyland",
                "time": "09:00",
                "duration": "Full day",
                "locations": [
                    "Hong Kong Disneyland"
                ]
            },
            {
                "name": "Lantau Island Tour",
                "time": "12:00",
                "duration": "6 hours",
                "locations": [
                    "Lantau Island",
                    "Ngong Ping Village",
                    "Po Lin Monastery",
                    "Grand Buddha Statue",
                    "Citygate Shopping Mall"
                ]
            },
            {
                "name": "Ocean Park",
                "time": "10:00",
                "duration": "Full day",
                "locations": [
                    "Ocean Park"
                ]
            }
        ]
    },

    "Shenzhen": {
        "days": 3,
        "activities": [
            {
                "name": "Shenzhen City Tour",
                "time": "10:00",
                "duration": "Full day",
                "locations": [
                    "Lotus Park",
                    "Jade Shop",
                    "Windows of the World",
                    "Louhu Shopping Mall"
                ]
            },
            {
                "name": "Free Day",
                "time": None,
                "duration": "Full day",
                "locations": []
            }
        ]
    },

    "Macau": {
        "days": 3,
        "activities": [
            {
                "name": "Macau City Tour",
                "time": "13:00",
                "duration": "4 hours",
                "locations": [
                    "Fishermen's Wharf",
                    "Ruins of St. Paul",
                    "Senado Square",
                    "Venetian Hotel",
                    "Londoner Hotel",
                    "Parisian Garden Eiffel Tower",
                    "Wynn Palace",
                    "Macau Tower"
                ]
            }
        ]
    }
}

print("City blocks created.")

for city, block in city_blocks.items():
    print(
        f"{city}: "
        f"{block['days']} days, "
        f"{len(block['activities'])} activities"
    )

In [ ]:
from itertools import permutations
from datetime import datetime, timedelta

START_DATE = datetime.strptime("2026-08-12", "%Y-%m-%d")
END_DATE = datetime.strptime("2026-08-21", "%Y-%m-%d")

# Number of calendar days
total_days = (END_DATE - START_DATE).days + 1

print("Total trip days:", total_days)

# Base allocation from the reference voucher
city_day_allocation = {
    "Hong Kong": 4,
    "Shenzhen": 3,
    "Macau": 2
}

print("City allocation:")
for city, days in city_day_allocation.items():
    print(f"{city}: {days} days")

print("Transition/arrival/departure days: included separately")

In [ ]:
from datetime import datetime, timedelta

START_DATE = datetime.strptime("2026-08-12", "%Y-%m-%d")
END_DATE = datetime.strptime("2026-08-21", "%Y-%m-%d")

city_day_allocation = {
    "Hong Kong": 4,
    "Shenzhen": 3,
    "Macau": 2
}


def allocate_city_dates(city_order):
    """
    Allocate trip dates according to the selected city order.
    """

    current_date = START_DATE
    schedule = []

    for city in city_order:

        number_of_days = city_day_allocation[city]

        city_dates = []

        for _ in range(number_of_days):

            if current_date <= END_DATE:
                city_dates.append(
                    current_date.strftime("%Y-%m-%d")
                )

                current_date += timedelta(days=1)

        schedule.append({
            "city": city,
            "dates": city_dates
        })

    return schedule


# Test Variation 1

variation_1 = allocate_city_dates(
    ["Hong Kong", "Shenzhen", "Macau"]
)

for block in variation_1:
    print(
        f"{block['city']}: "
        f"{block['dates']}"
    )
variation_2 = allocate_city_dates(
    ["Hong Kong", "Macau", "Shenzhen"]
)

for block in variation_2:
    print(
        f"{block['city']}: "
        f"{block['dates']}"
    )


In [ ]:
variation_2 = allocate_city_dates(
    ["Hong Kong", "Macau", "Shenzhen"]
)

for block in variation_2:
    print(
        f"{block['city']}: "
        f"{block['dates']}"
    )

In [ ]:
def build_activity_schedule(city_schedule, city_blocks):
    """
    Attach activities to the dates allocated to each city.
    """

    itinerary = []

    for block in city_schedule:

        city = block["city"]
        dates = block["dates"]
        activities = city_blocks[city]["activities"]

        city_days = []

        for date in dates:

            city_days.append({
                "date": date,
                "city": city,
                "activities": []
            })

        # Put activities into available city days
        for index, activity in enumerate(activities):

            if index < len(city_days):

                city_days[index]["activities"].append(
                    activity
                )

        itinerary.extend(city_days)

    return itinerary

In [ ]:
schedule_1 = allocate_city_dates(
    ["Hong Kong", "Shenzhen", "Macau"]
)

itinerary_1 = build_activity_schedule(
    schedule_1,
    city_blocks
)

for day in itinerary_1:

    print(
        f"\n{day['date']} — {day['city']}"
    )

    for activity in day["activities"]:

        print(
            f"  {activity['name']} "
            f"({activity['time']})"
        )

In [ ]:
transfer_options = {
    ("Hong Kong", "Shenzhen"): {
        "transport": "Train",
        "duration_hours": 2
    },

    ("Shenzhen", "Hong Kong"): {
        "transport": "Train",
        "duration_hours": 2
    },

    ("Hong Kong", "Macau"): {
        "transport": "Ferry",
        "duration_hours": 2
    },

    ("Macau", "Hong Kong"): {
        "transport": "Ferry",
        "duration_hours": 2
    },

    ("Shenzhen", "Macau"): {
        "transport": "Ferry",
        "duration_hours": 2
    },

    ("Macau", "Shenzhen"): {
        "transport": "Ferry",
        "duration_hours": 2
    }
}

print("Transfer options:", len(transfer_options))

In [ ]:
def create_transfers(city_order, city_schedule):
    """
    Create transfer events between consecutive cities.
    """

    transfers = []

    for i in range(len(city_order) - 1):

        from_city = city_order[i]
        to_city = city_order[i + 1]

        transfer = transfer_options.get(
            (from_city, to_city)
        )

        if transfer is None:
            raise ValueError(
                f"No transport defined for "
                f"{from_city} → {to_city}"
            )

        # Last date assigned to current city
        from_dates = city_schedule[i]["dates"]

        # First date assigned to next city
        to_dates = city_schedule[i + 1]["dates"]

        transfer_date = to_dates[0]

        transfers.append({
            "date": transfer_date,
            "from": from_city,
            "to": to_city,
            "transport": transfer["transport"],
            "duration_hours": transfer["duration_hours"]
        })

    return transfers

In [ ]:
city_order = [
    "Hong Kong",
    "Shenzhen",
    "Macau"
]

schedule = allocate_city_dates(
    city_order
)

transfers = create_transfers(
    city_order,
    schedule
)

for transfer in transfers:

    print(
        f"{transfer['date']}: "
        f"{transfer['from']} → "
        f"{transfer['to']} "
        f"({transfer['transport']}, "
        f"{transfer['duration_hours']} hours)"
    )

In [ ]:
def build_complete_itinerary(city_order):
    """
    Build one complete itinerary from:
    - city order
    - date allocation
    - activities
    - transfers
    """

    # Allocate dates
    city_schedule = allocate_city_dates(city_order)

    # Build activity schedule
    activity_itinerary = build_activity_schedule(
        city_schedule,
        city_blocks
    )

    # Build transfers
    transfers = create_transfers(
        city_order,
        city_schedule
    )

    # Index transfers by date
    transfer_by_date = {}

    for transfer in transfers:

        date = transfer["date"]

        if date not in transfer_by_date:
            transfer_by_date[date] = []

        transfer_by_date[date].append(
            transfer
        )

    # Add transfers to itinerary
    for day in activity_itinerary:

        date = day["date"]

        day["transfers"] = transfer_by_date.get(
            date,
            []
        )

    return activity_itinerary

In [ ]:
variation_1_itinerary = build_complete_itinerary(
    [
        "Hong Kong",
        "Shenzhen",
        "Macau"
    ]
)

for day in variation_1_itinerary:

    print(
        f"\n{day['date']} — {day['city']}"
    )

    for transfer in day["transfers"]:

        print(
            f"  TRANSFER: "
            f"{transfer['from']} → "
            f"{transfer['to']} "
            f"({transfer['transport']})"
        )

    for activity in day["activities"]:

        print(
            f"  ACTIVITY: "
            f"{activity['name']} "
            f"({activity['time']})"
        )

In [ ]:
timing_rules = {

    "arrival": {
        "date": "2026-08-12",
        "time": "15:30",
        "location": "Hong Kong International Airport"
    },

    "departure": {
        "date": "2026-08-21",
        "time": "07:30",
        "location": "HZMB Macau Port"
    },

    "minimum_transfer_buffer_minutes": 60,

    "fixed_activity_times": {
        "Hong Kong Night Tour": "17:00",
        "Hong Kong Disneyland": "09:00",
        "Lantau Island Tour": "12:00",
        "Ocean Park": "10:00",
        "Shenzhen City Tour": "10:00",
        "Macau City Tour": "13:00"
    }
}

print("Timing rules loaded.")

In [ ]:
activity_durations = {

    "Hong Kong Night Tour": 5 * 60,

    "Hong Kong Disneyland": 12 * 60,

    "Lantau Island Tour": 6 * 60,

    "Ocean Park": 9 * 60,

    "Shenzhen City Tour": 7 * 60,

    "Macau City Tour": 4 * 60,

    "Free Day": 0
}

print("Activity durations loaded.")

In [ ]:
from datetime import datetime, timedelta


def time_to_minutes(time_string):

    hour, minute = map(
        int,
        time_string.split(":")
    )

    return hour * 60 + minute


def validate_day(day):

    errors = []

    activities = day.get(
        "activities",
        []
    )

    transfers = day.get(
        "transfers",
        []
    )

    events = []

    # Add activities
    for activity in activities:

        name = activity["name"]

        start_time = activity.get(
            "time"
        )

        if start_time is None:
            continue

        duration = activity_durations.get(
            name,
            0
        )

        start = time_to_minutes(
            start_time
        )

        end = start + duration

        events.append({
            "type": "activity",
            "name": name,
            "start": start,
            "end": end
        })

    # Add transfers
    for transfer in transfers:

        # For now assign transfer at 08:00
        # on transition days.
        start = 8 * 60

        duration = (
            transfer["duration_hours"] * 60
        )

        end = start + duration

        events.append({
            "type": "transfer",
            "name": (
                f"{transfer['from']} → "
                f"{transfer['to']}"
            ),
            "start": start,
            "end": end
        })

    # Sort events
    events.sort(
        key=lambda x: x["start"]
    )

    # Detect overlap
    for i in range(
        len(events) - 1
    ):

        current = events[i]
        next_event = events[i + 1]

        if current["end"] > next_event["start"]:

            errors.append(
                f"OVERLAP: "
                f"{current['name']} "
                f"overlaps "
                f"{next_event['name']}"
            )

    return errors

In [ ]:
def schedule_transfer(
    transfer,
    start_time="08:00",
    buffer_minutes=60
):
    """
    Schedule a transfer and calculate when the next activity
    can safely begin.
    """

    start = time_to_minutes(start_time)

    duration = transfer["duration_hours"]

    arrival = start + duration

    next_available = (
        arrival + buffer_minutes
    )

    return {
        "start": start,
        "arrival": arrival,
        "next_available": next_available
    }

In [ ]:
for day in variation_1_itinerary:

    errors = validate_day(day)

    if errors:

        print(
            f"{day['date']} — INVALID"
        )

        for error in errors:
            print(
                " ",
                error
            )

    else:

        print(
            f"{day['date']} — OK"
        )

In [ ]:
transfer = {
    "from": "Hong Kong",
    "to": "Shenzhen",
    "transport": "Train",
    "duration_hours": 2
}

result = schedule_transfer(
    transfer,
    start_time="08:00",
    buffer_minutes=60
)

print(
    "Transfer starts:",
    result["start"]
)

print(
    "Arrival:",
    result["arrival"]
)

print(
    "Next activity can start:",
    result["next_available"]
)

In [ ]:
def minutes_to_time(minutes):
    """
    Convert minutes since midnight to HH:MM.
    """
    hours = minutes // 60
    mins = minutes % 60

    return f"{hours:02d}:{mins:02d}"


def get_activity_schedule_time(activity, day, occupied_events):
    """
    Find a valid start time for an activity.
    """

    name = activity["name"]

    # Free day requires no scheduling
    if name == "Free Day":
        return None

    duration = activity_durations.get(
        name,
        60
    )

    fixed_time = activity.get("time")

    # If activity has a fixed time,
    # try to use it first.
    if fixed_time:

        start = time_to_minutes(
            fixed_time
        )

    else:

        start = 9 * 60

    end = start + duration

    # Check against occupied events
    for event in occupied_events:

        if (
            start < event["end"]
            and end > event["start"]
        ):

            # Conflict detected.
            # Move activity after the conflicting event.
            start = event["end"]

            end = start + duration

    return {
        "start": minutes_to_time(start),
        "end": minutes_to_time(end),
        "duration_minutes": duration
    }

In [ ]:
def schedule_day(day):
    """
    Schedule transfers and activities for one day.

    Rules:
    - Transfers start at 08:00
    - A 60-minute buffer is required after a transfer
    - Activities cannot start before the transfer + buffer
    """

    scheduled_events = []

    TRANSFER_START = 8 * 60
    TRANSFER_BUFFER = 60

    # -------------------------
    # Schedule transfers
    # -------------------------

    for transfer in day.get("transfers", []):

        # Support duration in minutes
        duration = transfer.get("duration_minutes")

        # Backward compatibility with hours
        if duration is None:

            duration_hours = transfer.get(
                "duration_hours"
            )

            if duration_hours is not None:
                duration = duration_hours * 60

        if duration is None:
            raise ValueError(
                f"Transfer duration missing: {transfer}"
            )

        transfer_start = TRANSFER_START
        transfer_end = (
            transfer_start + duration
        )

        scheduled_events.append({
            "type": "transfer",
            "name": (
                f"{transfer['from']} → "
                f"{transfer['to']}"
            ),
            "start": transfer_start,
            "end": transfer_end
        })

    # -------------------------
    # Schedule activities
    # -------------------------

    scheduled_activities = []

    for activity in day.get("activities", []):

        timing = get_activity_schedule_time(
            activity,
            day,
            scheduled_events
        )

        if timing is None:

            scheduled_activities.append({
                **activity,
                "start": None,
                "end": None
            })

            continue

        # Convert scheduled time to minutes
        activity_start = time_to_minutes(
            timing["start"]
        )

        activity_end = time_to_minutes(
            timing["end"]
        )

        # ---------------------------------
        # STEP 17: Transfer buffer check
        # ---------------------------------

        for event in scheduled_events:

            if event["type"] != "transfer":
                continue

            earliest_start = (
                event["end"]
                + TRANSFER_BUFFER
            )

            if activity_start < earliest_start:

                duration = (
                    activity_end
                    - activity_start
                )

                activity_start = earliest_start

                activity_end = (
                    activity_start
                    + duration
                )

        scheduled_event = {
            "type": "activity",
            "name": activity["name"],
            "start": activity_start,
            "end": activity_end
        }

        scheduled_events.append(
            scheduled_event
        )

        scheduled_activities.append({
            **activity,
            "start": minutes_to_time(
                activity_start
            ),
            "end": minutes_to_time(
                activity_end
            )
        })

    return scheduled_activities

In [ ]:
print(get_activity_schedule_time)

In [ ]:
for day in variation_1_itinerary:

    if day["date"] == "2026-08-16":

        scheduled = schedule_day(day)

        print(
            f"{day['date']} — {day['city']}"
        )

        for event in scheduled:

            print(
                f"{event['name']}: "
                f"{event['start']} → "
                f"{event['end']}"
            )

In [ ]:

for day in variation_1_itinerary:
    print("\nDATE:", day["date"])
    print("CITY:", day.get("city"))

    for activity in day.get("activities", []):
        print(activity)

In [ ]:
def normalize_duration(duration):
    """
    Convert voucher duration strings into minutes.
    """

    if duration is None:
        return None

    duration_lower = duration.lower().strip()

    # Explicit hours
    if "hour" in duration_lower:
        try:
            hours = float(duration_lower.split()[0])
            return int(hours * 60)
        except:
            return None

    # Full day
    if duration_lower == "full day":
        return 12 * 60

    return None

In [ ]:
test_durations = [
    "5 hours",
    "6 hours",
    "4 hours",
    "Full day",
    None
]

for duration in test_durations:
    print(duration, "→", normalize_duration(duration), "minutes")

In [ ]:
for day in variation_1_itinerary:

    for activity in day.get("activities", []):

        activity["duration_minutes"] = normalize_duration(
            activity.get("duration")
        )

In [ ]:
for day in variation_1_itinerary:

    for activity in day.get("activities", []):

        print(
            activity["name"],
            "→",
            activity.get("duration"),
            "→",
            activity.get("duration_minutes"),
            "minutes"
        )

In [ ]:
for day in variation_1_itinerary:

    for activity in day.get("activities", []):

        print(
            activity["name"],
            "| time:",
            activity.get("time"),
            "| duration:",
            activity.get("duration")
        )

In [ ]:
def normalize_duration(duration):
    """
    Normalize voucher duration.

    Explicit durations are converted to minutes.
    Full-day activities are marked separately.
    """

    if duration is None:
        return {
            "duration_type": None,
            "duration_minutes": None
        }

    duration_lower = duration.lower().strip()

    if duration_lower == "full day":
        return {
            "duration_type": "full_day",
            "duration_minutes": None
        }

    if "hour" in duration_lower:
        try:
            hours = float(duration_lower.split()[0])

            return {
                "duration_type": "fixed",
                "duration_minutes": int(hours * 60)
            }

        except (ValueError, IndexError):
            pass

    return {
        "duration_type": "unknown",
        "duration_minutes": None
    }

In [ ]:
for day in variation_1_itinerary:

    for activity in day.get("activities", []):

        normalized = normalize_duration(
            activity.get("duration")
        )

        activity["duration_type"] = normalized["duration_type"]
        activity["duration_minutes"] = normalized["duration_minutes"]

In [ ]:
def validate_day(day, scheduled_activities):

    errors = []

    city = day.get("city")
    transfers = day.get("transfers", [])

    # 1. Validate activities
    for activity in scheduled_activities:

        start = activity.get("start")
        end = activity.get("end")

        if activity.get("duration_type") != "full_day":

            if start is None or end is None:
                errors.append(
                    f"{activity['name']}: missing schedule"
                )
                continue

            start_minutes = time_to_minutes(start)
            end_minutes = time_to_minutes(end)

            if end_minutes <= start_minutes:
                errors.append(
                    f"{activity['name']}: "
                    f"end time must be after start time"
                )

        activity_city = activity.get("city")

        if activity_city is not None and activity_city != city:
            errors.append(
                f"{activity['name']}: "
                f"activity city '{activity_city}' "
                f"does not match day city '{city}'"
            )

    # 2. Check overlaps
    ...

    # 3. Transfer buffer
    ...

    return {
        "valid": len(errors) == 0,
        "errors": errors
    }

In [ ]:
scheduled = []

for day in variation_1_itinerary:

    result = schedule_day(day)

    validation = validate_day(
        day,
        result
    )

    print(
        day["date"],
        day.get("city"),
        "→",
        validation["valid"]
    )

    if not validation["valid"]:
        print(
            "Errors:",
            validation["errors"]
        )

In [ ]:
CANONICAL_SCHEMA_VERSION = "1.0"


def create_canonical_itinerary(
    voucher_id,
    original_days,
    scheduled_days
):
    itinerary = {
        "schema_version": CANONICAL_SCHEMA_VERSION,
        "voucher_id": voucher_id,
        "days": []
    }

    for original_day, scheduled_activities in zip(
        original_days,
        scheduled_days
    ):

        canonical_day = {
            "date": original_day.get("date"),
            "city": original_day.get("city"),
            "activities": [],
            "transfers": []
        }

        # Activities
        for activity in scheduled_activities:

            canonical_activity = {
                "name": activity.get("name"),
                "city": original_day.get("city"),

                "requested_start": activity.get("time"),

                "start": activity.get("start"),
                "end": activity.get("end"),

                "duration_minutes": activity.get(
                    "duration_minutes"
                ),

                "duration_type": activity.get(
                    "duration_type"
                ),

                "locations": activity.get(
                    "locations",
                    []
                )
            }

            canonical_day["activities"].append(
                canonical_activity
            )

        # Transfers
        for transfer in original_day.get(
            "transfers",
            []
        ):

            canonical_transfer = {
                "from": transfer.get("from"),
                "to": transfer.get("to"),
                "duration_minutes": (
                    transfer.get(
                        "duration_hours",
                        0
                    ) * 60
                ),
                "type": transfer.get(
                    "type",
                    "transfer"
                )
            }

            canonical_day["transfers"].append(
                canonical_transfer
            )

        itinerary["days"].append(canonical_day)

    return itinerary

In [ ]:
scheduled_days = []

for day in variation_1_itinerary:

    result = schedule_day(day)

    scheduled_days.append(result)

print("Scheduled days generated:", len(scheduled_days))

In [ ]:
canonical_itinerary = create_canonical_itinerary(
    voucher_id="voucher_001",
    original_days=variation_1_itinerary,
    scheduled_days=scheduled_days
)

print("Canonical itinerary created.")

In [ ]:
import pprint

pprint.pprint(
    canonical_itinerary,
    sort_dicts=False
)

In [ ]:
for day in canonical_itinerary["days"]:
    for activity in day["activities"]:
        print(
            day["date"],
            "|",
            activity["name"],
            "| requested:",
            activity["requested_start"],
            "| scheduled:",
            activity["start"],
            "→",
            activity["end"]
        )

In [ ]:
from itertools import permutations

In [ ]:
def generate_activity_permutations(activities):
    """
    Generate different activity orders.

    Returns:
        List of activity-order variations.
    """

    # No activities
    if not activities:
        return [[]]

    # One activity
    if len(activities) == 1:
        return [activities]

    # Generate permutations
    return [
        list(order)
        for order in permutations(activities)
    ]

In [ ]:
for day in variation_1_itinerary:

    activities = day.get("activities", [])

    variations = generate_activity_permutations(
        activities
    )

    print(
        day["date"],
        day["city"],
        "→",
        len(variations),
        "possible orders"
    )

In [ ]:
def generate_activity_permutations(activities):
    """
    Generate unique activity orderings.

    The original ordering is always included first.
    """

    if not activities:
        return [[]]

    if len(activities) == 1:
        return [activities]

    original = list(activities)

    all_orders = [
        list(order)
        for order in permutations(activities)
    ]

    # Remove duplicates while preserving order
    unique_orders = []

    seen = set()

    for order in all_orders:

        key = tuple(
            activity["name"]
            for activity in order
        )

        if key not in seen:
            seen.add(key)
            unique_orders.append(order)

    # Put original order first
    original_key = tuple(
        activity["name"]
        for activity in original
    )

    unique_orders.sort(
        key=lambda order:
        tuple(
            activity["name"]
            for activity in order
        ) != original_key
    )

    return unique_orders

In [ ]:
print("Testing Step 21.1")
print("variation_1_itinerary exists:", "variation_1_itinerary" in globals())

In [ ]:
for day in variation_1_itinerary:

    activities = day.get("activities", [])

    variations = generate_activity_permutations(
        activities
    )

    print(
        day["date"],
        "|",
        day["city"],
        "|",
        len(activities),
        "activities |",
        len(variations),
        "possible orders"
    )

In [ ]:
test_activities = [
    {"name": "Activity A"},
    {"name": "Activity B"},
    {"name": "Activity C"}
]

test_variations = generate_activity_permutations(
    test_activities
)

print("Number of variations:", len(test_variations))

for i, variation in enumerate(test_variations, 1):

    print(
        i,
        "→",
        [activity["name"] for activity in variation]
    )

In [ ]:
from itertools import product

In [ ]:
def generate_itinerary_order_variations(itinerary):
    """
    Generate complete itinerary variations by
    combining activity-order variations across days.

    Dates and cities remain unchanged.
    Only activity ordering changes.
    """

    day_variations = []

    for day in itinerary:

        activities = day.get(
            "activities",
            []
        )

        variations = generate_activity_permutations(
            activities
        )

        day_variations.append(
            variations
        )

    # Combine one variation from each day
    complete_variations = []

    for combination in product(
        *day_variations
    ):

        new_itinerary = []

        for day, activities in zip(
            itinerary,
            combination
        ):

            new_day = day.copy()

            new_day["activities"] = activities

            new_itinerary.append(
                new_day
            )

        complete_variations.append(
            new_itinerary
        )

    return complete_variations

In [ ]:
itinerary_variations = (
    generate_itinerary_order_variations(
        variation_1_itinerary
    )
)

print(
    "Complete itinerary variations:",
    len(itinerary_variations)
)

In [ ]:
test_itinerary = [

    {
        "date": "2026-08-12",
        "city": "Hong Kong",
        "activities": [
            {"name": "Activity A"},
            {"name": "Activity B"}
        ],
        "transfers": []
    },

    {
        "date": "2026-08-13",
        "city": "Hong Kong",
        "activities": [
            {"name": "Activity C"},
            {"name": "Activity D"},
            {"name": "Activity E"}
        ],
        "transfers": []
    }
]

In [ ]:
test_variations = (
    generate_itinerary_order_variations(
        test_itinerary
    )
)

print(
    "Complete variations:",
    len(test_variations)
)

In [ ]:
for i, itinerary in enumerate(
    test_variations[:5],
    1
):

    print(f"\nVariation {i}")

    for day in itinerary:

        print(
            day["date"],
            "→",
            [
                activity["name"]
                for activity in day["activities"]
            ]
        )

In [ ]:
def validate_itinerary_variation(
    itinerary
):
    """
    Schedule and validate every day
    of an itinerary variation.

    Returns:
        {
            "valid": True/False,
            "scheduled_days": [...],
            "errors": [...]
        }
    """

    scheduled_days = []
    errors = []

    for day in itinerary:

        # Schedule activities for this day
        scheduled_activities = schedule_day(day)

        # Validate the scheduled day
        validation = validate_day(
            day,
            scheduled_activities
        )

        scheduled_days.append(
            scheduled_activities
        )

        if not validation["valid"]:

            errors.extend(
                [
                    f"{day['date']}: {error}"
                    for error in validation["errors"]
                ]
            )

    return {
        "valid": len(errors) == 0,
        "scheduled_days": scheduled_days,
        "errors": errors
    }

In [ ]:
result = validate_itinerary_variation(
    variation_1_itinerary
)

print(
    "Valid:",
    result["valid"]
)

if not result["valid"]:

    print("Errors:")

    for error in result["errors"]:
        print("-", error)

In [ ]:
for i, variation in enumerate(
    test_variations,
    1
):

    result = validate_itinerary_variation(
        variation
    )

    print(
        f"Variation {i}:",
        result["valid"]
    )

    if not result["valid"]:

        print(
            "Errors:",
            result["errors"]
        )

In [ ]:
def build_valid_canonical_variations(
    itinerary_variations,
    voucher_id
):
    """
    Schedule, validate and canonicalize
    itinerary variations.

    Invalid variations are discarded.
    """

    valid_variations = []

    for variation in itinerary_variations:

        result = validate_itinerary_variation(
            variation
        )

        # Reject invalid variations
        if not result["valid"]:
            continue

        # Convert to canonical format
        canonical = create_canonical_itinerary(
            voucher_id=voucher_id,
            original_days=variation,
            scheduled_days=result["scheduled_days"]
        )

        valid_variations.append(
            canonical
        )

    return valid_variations

In [ ]:
valid_canonical_variations = (
    build_valid_canonical_variations(
        itinerary_variations=[
            variation_1_itinerary
        ],
        voucher_id="voucher_001"
    )
)

print(
    "Valid canonical variations:",
    len(valid_canonical_variations)
)

In [ ]:
valid_test_variations = (
    build_valid_canonical_variations(
        itinerary_variations=test_variations,
        voucher_id="test_voucher"
    )
)

print(
    "Valid test variations:",
    len(valid_test_variations)
)

In [ ]:
def itinerary_signature(itinerary):
    """
    Create a unique, hashable representation
    of an itinerary.
    """

    signature = []

    for day in itinerary["days"]:

        day_signature = (
            day["date"],
            day["city"],
            tuple(
                (
                    activity["name"],
                    activity.get("start"),
                    activity.get("end"),
                    activity.get("duration_minutes"),
                    activity.get("duration_type")
                )
                for activity in day.get(
                    "activities",
                    []
                )
            )
        )

        signature.append(day_signature)

    return tuple(signature)

In [ ]:
def deduplicate_itineraries(itineraries):
    """
    Remove duplicate canonical itineraries.
    """

    unique_itineraries = []

    seen = set()

    for itinerary in itineraries:

        signature = itinerary_signature(
            itinerary
        )

        if signature in seen:
            continue

        seen.add(signature)

        unique_itineraries.append(
            itinerary
        )

    return unique_itineraries

In [ ]:
unique_test_variations = (
    deduplicate_itineraries(
        valid_test_variations
    )
)

print(
    "Before deduplication:",
    len(valid_test_variations)
)

print(
    "After deduplication:",
    len(unique_test_variations)
)

In [ ]:
duplicate_test = (
    valid_test_variations
    + valid_test_variations[:3]
)

print(
    "Before:",
    len(duplicate_test)
)

unique_duplicate_test = (
    deduplicate_itineraries(
        duplicate_test
    )
)

print(
    "After:",
    len(unique_duplicate_test)
)

In [ ]:
def calculate_itinerary_score(itinerary):
    """
    Calculate a basic quality score for a canonical itinerary.

    Returns a score from 0 to 100.
    """

    score = 100

    total_activities = 0
    total_gaps = 0

    for day in itinerary["days"]:

        activities = day.get(
            "activities",
            []
        )

        # Count activities
        total_activities += len(activities)

        # --------------------------------
        # Calculate gaps
        # --------------------------------

        timed_activities = [
            activity
            for activity in activities
            if activity.get("start") is not None
            and activity.get("end") is not None
        ]

        timed_activities.sort(
            key=lambda activity:
            time_to_minutes(activity["start"])
        )

        for i in range(
            len(timed_activities) - 1
        ):

            current_end = time_to_minutes(
                timed_activities[i]["end"]
            )

            next_start = time_to_minutes(
                timed_activities[i + 1]["start"]
            )

            gap = next_start - current_end

            if gap > 0:
                total_gaps += gap

    # --------------------------------
    # Penalize excessive gaps
    # --------------------------------

    gap_penalty = total_gaps / 60

    score -= gap_penalty * 2

    # Keep score between 0 and 100
    score = max(
        0,
        min(
            100,
            score
        )
    )

    return round(
        score,
        2
    )

In [ ]:
score = calculate_itinerary_score(
    canonical_itinerary
)

print(
    "Itinerary score:",
    score
)

In [ ]:
def calculate_day_coverage(day):
    """
    Calculate the percentage of the usable day
    occupied by timed activities.

    Returns:
        coverage percentage from 0 to 100
    """

    activities = day.get("activities", [])

    timed_activities = [
        activity
        for activity in activities
        if activity.get("start") is not None
        and activity.get("end") is not None
    ]

    if not timed_activities:
        return 0

    total_activity_minutes = 0

    for activity in timed_activities:

        start = time_to_minutes(
            activity["start"]
        )

        end = time_to_minutes(
            activity["end"]
        )

        duration = end - start

        if duration > 0:
            total_activity_minutes += duration

    earliest_start = min(
        time_to_minutes(
            activity["start"]
        )
        for activity in timed_activities
    )

    latest_end = max(
        time_to_minutes(
            activity["end"]
        )
        for activity in timed_activities
    )

    usable_minutes = (
        latest_end - earliest_start
    )

    if usable_minutes <= 0:
        return 0

    coverage = (
        total_activity_minutes
        / usable_minutes
    ) * 100

    return round(
        coverage,
        2
    )

In [ ]:
for day in canonical_itinerary["days"]:

    coverage = calculate_day_coverage(
        day
    )

    print(
        day["date"],
        "|",
        day["city"],
        "| Coverage:",
        coverage,
        "%"
    )

In [ ]:
def calculate_day_utilization(day):
    """
    Calculate how much of a standard usable day
    is occupied by activities.

    Usable day:
        08:00 → 22:00

    Returns:
        utilization percentage from 0 to 100
    """

    DAY_START = 8 * 60
    DAY_END = 22 * 60

    usable_minutes = (
        DAY_END - DAY_START
    )

    activities = day.get(
        "activities",
        []
    )

    total_activity_minutes = 0

    for activity in activities:

        start = activity.get("start")
        end = activity.get("end")

        if start is None or end is None:
            continue

        start_minutes = time_to_minutes(
            start
        )

        end_minutes = time_to_minutes(
            end
        )

        # Clamp activity to usable day
        start_minutes = max(
            start_minutes,
            DAY_START
        )

        end_minutes = min(
            end_minutes,
            DAY_END
        )

        if end_minutes > start_minutes:

            total_activity_minutes += (
                end_minutes - start_minutes
            )

    utilization = (
        total_activity_minutes
        / usable_minutes
    ) * 100

    return round(
        min(utilization, 100),
        2
    )

In [ ]:
for day in canonical_itinerary["days"]:

    utilization = calculate_day_utilization(
        day
    )

    print(
        day["date"],
        "|",
        day["city"],
        "| Utilization:",
        utilization,
        "%"
    )

In [ ]:
def calculate_transfer_efficiency(day):
    """
    Calculate transfer efficiency for one day.

    Returns:
        100 when there are no transfers to evaluate.
        Lower values when transfer duration is high.
    """

    transfers = day.get("transfers", [])

    if not transfers:
        return 100.0

    total_transfer_minutes = 0

    for transfer in transfers:

        duration = transfer.get(
            "duration_minutes",
            0
        )

        if duration is not None and duration > 0:
            total_transfer_minutes += duration

    if total_transfer_minutes == 0:
        return 100.0

    # Simple initial penalty:
    # every 60 minutes of transfer time
    # reduces efficiency by 10 points.

    penalty = (
        total_transfer_minutes / 60
    ) * 10

    score = 100 - penalty

    return round(
        max(0, score),
        2
    )

In [ ]:
for day in canonical_itinerary["days"]:

    efficiency = calculate_transfer_efficiency(
        day
    )

    print(
        day["date"],
        "|",
        day["city"],
        "| Transfer efficiency:",
        efficiency
    )

In [ ]:
def extract_day_features(day):
    """
    Extract ML features for one itinerary day.
    """

    return {
        "date": day.get("date"),
        "city": day.get("city"),

        "activity_count": len(
            day.get("activities", [])
        ),

        "coverage": calculate_day_coverage(
            day
        ),

        "utilization": calculate_day_utilization(
            day
        ),

        "transfer_efficiency": calculate_transfer_efficiency(
            day
        )
    }

In [ ]:
for day in canonical_itinerary["days"]:

    features = extract_day_features(
        day
    )

    print(features)

In [ ]:
def extract_itinerary_features(
    itinerary,
    travel_times=None
):
    """
    Extract aggregate ML features
    for a complete itinerary.
    """

    days = itinerary.get(
        "days",
        []
    )

    day_features = [
        extract_day_features(day)
        for day in days
    ]

    if not day_features:
        return {
            "total_days": 0,
            "total_activities": 0,
            "average_coverage": 0,
            "average_utilization": 0,
            "average_transfer_efficiency": 0,
            "average_travel_efficiency": 0,
            "total_travel_time": 0,
            "free_days": 0
        }

    total_days = len(day_features)

    total_activities = sum(
        day["activity_count"]
        for day in day_features
    )

    average_coverage = (
        sum(
            day["coverage"]
            for day in day_features
        )
        / total_days
    )

    average_utilization = (
        sum(
            day["utilization"]
            for day in day_features
        )
        / total_days
    )

    average_transfer_efficiency = (
        sum(
            day["transfer_efficiency"]
            for day in day_features
        )
        / total_days
    )

    # --------------------------------
    # Travel efficiency
    # --------------------------------

    if travel_times is None:

        average_travel_efficiency = 100.0

    else:

        travel_efficiencies = [
            calculate_travel_efficiency(
                day,
                travel_times
            )
            for day in days
        ]

        average_travel_efficiency = (
            sum(travel_efficiencies)
            / total_days
        )

    # --------------------------------
    # Total travel time
    # --------------------------------

    if travel_times is None:

        total_travel_time = 0

    else:

        total_travel_time = sum(
            calculate_total_travel_time(
                day,
                travel_times
            )
            for day in days
        )

    # --------------------------------
    # Free days
    # --------------------------------

    free_days = sum(
        1
        for day in day_features
        if day["activity_count"] == 0
    )

    return {
        "total_days": total_days,
        "total_activities": total_activities,
        "average_coverage": round(
            average_coverage,
            2
        ),
        "average_utilization": round(
            average_utilization,
            2
        ),
        "average_transfer_efficiency": round(
            average_transfer_efficiency,
            2
        ),
        "average_travel_efficiency": round(
            average_travel_efficiency,
            2
        ),
        "total_travel_time": total_travel_time,
        "free_days": free_days
    }

In [ ]:
itinerary_features = extract_itinerary_features(
    canonical_itinerary
)

print(itinerary_features)

In [ ]:
def calculate_final_itinerary_score(
    itinerary,
    travel_times=None
):
    """
    Calculate the overall quality score
    for a complete itinerary.

    Score range: 0-100.
    """

    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    coverage_score = features[
        "average_coverage"
    ]

    utilization_score = features[
        "average_utilization"
    ]

    transfer_score = features[
        "average_transfer_efficiency"
    ]

    travel_score = features[
        "average_travel_efficiency"
    ]

    activity_score = min(
        features["total_activities"] * 10,
        100
    )

    final_score = (
        coverage_score * 0.25
        + utilization_score * 0.25
        + transfer_score * 0.20
        + travel_score * 0.15
        + activity_score * 0.15
    )

    final_score = max(
        0,
        min(
            100,
            final_score
        )
    )

    return round(
        final_score,
        2
    )

In [ ]:
final_score = calculate_final_itinerary_score(
    canonical_itinerary
)

print(
    "Final itinerary score:",
    final_score
)

In [ ]:
def rank_itineraries(itineraries):
    """
    Score and rank valid canonical itineraries.
    """

    ranked = []

    for itinerary in itineraries:

        score = calculate_final_itinerary_score(
            itinerary
        )

        ranked.append({
            "itinerary": itinerary,
            "score": score
        })

    ranked.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return ranked

In [ ]:
ranked_test_variations = rank_itineraries(
    unique_test_variations
)

for i, item in enumerate(
    ranked_test_variations,
    1
):

    print(
        f"Rank {i} | "
        f"Score: {item['score']}"
    )

In [ ]:
def calculate_sequence_features(day):
    """
    Extract sequence-related features from a day.
    """

    activities = day.get(
        "activities",
        []
    )

    if len(activities) <= 1:
        return {
            "activity_transitions": 0
        }

    return {
        "activity_transitions": len(activities) - 1
    }

In [ ]:
for day in canonical_itinerary["days"]:

    features = calculate_sequence_features(
        day
    )

    print(
        day["date"],
        "|",
        day["city"],
        "|",
        features
    )

In [ ]:
for day in canonical_itinerary["days"]:

    for activity in day.get(
        "activities",
        []
    ):

        print(
            activity["name"],
            "|",
            activity.get("locations", [])
        )

In [ ]:
def extract_location_features(activity):
    """
    Extract location information from an activity.
    """

    locations = activity.get(
        "locations",
        []
    )

    return {
        "location_count": len(locations),
        "has_location": len(locations) > 0
    }

In [ ]:
for day in canonical_itinerary["days"]:

    for activity in day.get(
        "activities",
        []
    ):

        print(
            activity["name"],
            "|",
            extract_location_features(
                activity
            )
        )

In [ ]:
def calculate_activity_transitions(day):
    """
    Extract transitions between consecutive activities.
    """

    activities = day.get(
        "activities",
        []
    )

    transitions = []

    for i in range(len(activities) - 1):

        current = activities[i]
        next_activity = activities[i + 1]

        transitions.append({
            "from": current["name"],
            "to": next_activity["name"]
        })

    return transitions

In [ ]:
for day in canonical_itinerary["days"]:

    transitions = calculate_activity_transitions(
        day
    )

    print(
        day["date"],
        "|",
        transitions
    )

In [ ]:
test_day = {
    "date": "2026-08-21",
    "city": "Hong Kong",
    "activities": [
        {
            "name": "Activity A",
            "locations": ["Location A"],
            "start": "09:00",
            "end": "12:00"
        },
        {
            "name": "Activity B",
            "locations": ["Location B"],
            "start": "13:00",
            "end": "16:00"
        },
        {
            "name": "Activity C",
            "locations": ["Location C"],
            "start": "17:00",
            "end": "20:00"
        }
    ]
}

In [ ]:
print(
    calculate_activity_transitions(
        test_day
    )
)

In [ ]:
def get_travel_time(
    from_activity,
    to_activity,
    travel_times
):
    """
    Get travel time between two activities.

    Returns:
        travel time in minutes
    """

    return travel_times.get(
        (
            from_activity,
            to_activity
        )
    )

In [ ]:
travel_times = {
    ("Activity A", "Activity B"): 30,
    ("Activity B", "Activity C"): 45
}

print(
    get_travel_time(
        "Activity A",
        "Activity B",
        travel_times
    )
)

print(
    get_travel_time(
        "Activity B",
        "Activity C",
        travel_times
    )
)

In [ ]:
print(
    get_travel_time(
        "Activity A",
        "Activity C",
        travel_times
    )
)

In [ ]:
def calculate_total_travel_time(
    day,
    travel_times
):
    """
    Calculate total travel time between
    consecutive activities.

    Returns:
        total travel time in minutes
    """

    transitions = calculate_activity_transitions(
        day
    )

    total_travel_time = 0

    for transition in transitions:

        travel_time = get_travel_time(
            transition["from"],
            transition["to"],
            travel_times
        )

        # Unknown travel time
        if travel_time is None:
            continue

        total_travel_time += travel_time

    return total_travel_time

In [ ]:
travel_times = {
    ("Activity A", "Activity B"): 30,
    ("Activity B", "Activity C"): 45
}

total = calculate_total_travel_time(
    test_day,
    travel_times
)

print(
    "Total travel time:",
    total,
    "minutes"
)

In [ ]:
def calculate_travel_efficiency(
    day,
    travel_times
):
    """
    Calculate travel efficiency for a day.

    Returns a score between 0 and 100.
    """

    activities = day.get(
        "activities",
        []
    )

    if len(activities) <= 1:
        return 100.0

    total_travel_time = calculate_total_travel_time(
        day,
        travel_times
    )

    # No known travel time
    if total_travel_time == 0:
        return 100.0

    activity_count = len(activities)

    average_travel_time = (
        total_travel_time
        / (activity_count - 1)
    )

    # Initial heuristic
    efficiency = max(
        0,
        100 - average_travel_time
    )

    return round(
        efficiency,
        2
    )

In [ ]:
travel_times = {
    ("Activity A", "Activity B"): 30,
    ("Activity B", "Activity C"): 45
}

efficiency = calculate_travel_efficiency(
    test_day,
    travel_times
)

print(
    "Travel efficiency:",
    efficiency
)

In [ ]:
features = extract_itinerary_features(
    canonical_itinerary,
    travel_times={}
)

print(features)

In [ ]:
ITINERARY_FEATURE_ORDER = [
    "total_days",
    "total_activities",
    "average_coverage",
    "average_utilization",
    "average_transfer_efficiency",
    "average_travel_efficiency",
    "total_travel_time",
    "free_days"
]


def create_feature_vector(features):
    """
    Convert itinerary feature dictionary
    into a fixed-order numerical vector.
    """

    return [
        features[name]
        for name in ITINERARY_FEATURE_ORDER
    ]

In [ ]:
feature_vector = create_feature_vector(
    features
)

print(feature_vector)

In [ ]:
def prepare_ml_features(itinerary, travel_times=None):
    """
    Prepare complete ML feature representation
    for one itinerary.
    """

    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    vector = create_feature_vector(
        features
    )

    return {
        "features": features,
        "vector": vector
    }

In [ ]:
ml_features = prepare_ml_features(
    canonical_itinerary,
    travel_times={}
)

print("Features:")
print(ml_features["features"])

print("\nVector:")
print(ml_features["vector"])

In [ ]:
FEATURE_SCHEMA = {
    "version": "1.0",
    "features": [
        {
            "name": "total_days",
            "type": "numeric"
        },
        {
            "name": "total_activities",
            "type": "numeric"
        },
        {
            "name": "average_coverage",
            "type": "numeric"
        },
        {
            "name": "average_utilization",
            "type": "numeric"
        },
        {
            "name": "average_transfer_efficiency",
            "type": "numeric"
        },
        {
            "name": "average_travel_efficiency",
            "type": "numeric"
        },
        {
            "name": "free_days",
            "type": "numeric"
        }
    ]
}

print(FEATURE_SCHEMA)

In [ ]:
print(
    [
        feature["name"]
        for feature in FEATURE_SCHEMA["features"]
    ]
)

print(ITINERARY_FEATURE_ORDER)

In [ ]:
def create_training_target(itinerary):
    """
    Temporary quality target for training-data generation.

    Returns a score between 0 and 100.
    """

    return calculate_final_itinerary_score(
        itinerary
    )

In [ ]:
target = create_training_target(
    canonical_itinerary
)

print(
    "Training target:",
    target
)

In [ ]:
def create_training_record(
    itinerary,
    travel_times=None
):
    """
    Create one ML training record.
    """

    ml_features = prepare_ml_features(
        itinerary,
        travel_times=travel_times
    )

    target = create_training_target(
        itinerary
    )

    return {
        "X": ml_features["vector"],
        "y": target
    }

In [ ]:
training_record = create_training_record(
    canonical_itinerary,
    travel_times={}
)

print(training_record)

In [ ]:
def create_training_records(
    itineraries,
    travel_times=None
):
    """
    Create ML training records from
    multiple itinerary variations.
    """

    records = []

    for itinerary in itineraries:

        record = create_training_record(
            itinerary,
            travel_times=travel_times
        )

        records.append(record)

    return records

In [ ]:
training_records = create_training_records(
    unique_test_variations,
    travel_times={}
)

print(
    "Number of training records:",
    len(training_records)
)

In [ ]:
for i, record in enumerate(
    training_records[:5],
    1
):

    print(
        f"Record {i}:",
        record
    )

In [ ]:
test_itinerary = {
    "days": [test_day]
}

In [ ]:
features = extract_itinerary_features(
    test_itinerary,
    travel_times=travel_times
)

print(features)

vector = create_feature_vector(features)

print(vector)

In [ ]:
# Step 22.1
# Inspect the structure of generated variations

print("Type of variations:", type(variations))
print("Number of variations:", len(variations))

if len(variations) > 0:
    print("\nType of first variation:", type(variations[0]))
    print("\nFirst variation:")
    print(variations[0])

In [ ]:
# Step 22.1
# Inspect the structure of the ranked itinerary variations

print("Number of ranked variations:", len(ranked_test_variations))

print("\nType of first variation:")
print(type(ranked_test_variations[0]))

print("\nFirst ranked variation:")
print(ranked_test_variations[0])

In [ ]:
# Step 22.1
# Build ML dataset from ranked and validated itinerary variations

X = []
y = []

for item in ranked_test_variations:

    itinerary = item["itinerary"]
    score = item["score"]

    # Extract itinerary features
    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    # Convert features into ordered numerical vector
    vector = create_feature_vector(features)

    X.append(vector)
    y.append(score)

print("Total ML samples:", len(X))
print("Features per sample:", len(X[0]) if X else 0)

print("\nFirst feature vector:")
print(X[0])

print("\nFirst target score:")
print(y[0])

In [ ]:
# Step 22.2
# Convert ML dataset to NumPy arrays

import numpy as np

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Step 23.1
# Split dataset into training and testing sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# Step 23.2
# Scale the features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("\nScaled training data:")
print(X_train_scaled)

print("\nScaled testing data:")
print(X_test_scaled)

In [ ]:
# Step 24.1
# Train the first ML model

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=5
)

model.fit(X_train_scaled, y_train)

print("Model training completed.")

In [ ]:
# Step 24.2
# Predict itinerary scores on the test set

y_pred = model.predict(X_test_scaled)

print("Actual scores:")
print(y_test)

print("\nPredicted scores:")
print(y_pred)

In [ ]:
# Step 25
# Evaluate the baseline model

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation")
print("-----------------")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
# Step 26
# Analyze feature and target diversity

print("===== DATASET DIVERSITY ANALYSIS =====")

print("\nTotal samples:", len(X))

print("\nUnique target scores:")
print(np.unique(y))

print("\nNumber of unique target scores:", len(np.unique(y)))

print("\nUnique values per feature:")

for i, feature_name in enumerate(ITINERARY_FEATURE_ORDER):
    unique_values = np.unique(X[:, i])

    print(
        f"{i}. {feature_name}: "
        f"{len(unique_values)} unique value(s) → "
        f"{unique_values}"
    )

In [ ]:
# Step 27.1
# Check whether our generated variations are genuinely different

print("Total ranked variations:", len(ranked_test_variations))

print("\nVariation scores:")

for i, item in enumerate(ranked_test_variations, start=1):
    print(
        f"Variation {i}: "
        f"Score = {item['score']}"
    )

print("\nUnique scores:")
print(
    sorted(
        set(item["score"] for item in ranked_test_variations)
    )
)

In [ ]:
# Step 27.1b
# Inspect activity sequences

for i, item in enumerate(ranked_test_variations, start=1):

    itinerary = item["itinerary"]

    activities = []

    for day in itinerary["days"]:
        for activity in day["activities"]:
            activities.append(activity["name"])

    print(f"Variation {i}: {activities}")

In [ ]:
# Step 27.1
# Compare activity order, features and score

for i, item in enumerate(ranked_test_variations, start=1):

    itinerary = item["itinerary"]
    score = item["score"]

    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    print(f"\nVariation {i}")
    print("-" * 40)
    print("Score:", score)
    print("Features:", features)

In [ ]:
# Step 27.2
# Inspect the current scoring functions

import inspect

print("===== calculate_itinerary_score =====")
print(inspect.getsource(calculate_itinerary_score))

print("\n===== calculate_final_itinerary_score =====")
print(inspect.getsource(calculate_final_itinerary_score))

In [ ]:
# Step 27.4
# Recalculate scores using the updated scoring function

for i, item in enumerate(ranked_test_variations, start=1):

    itinerary = item["itinerary"]

    new_score = calculate_final_itinerary_score(
        itinerary
    )

    print(
        f"Variation {i}: "
        f"Old Score = {item['score']}, "
        f"New Score = {new_score}"
    )

In [ ]:
# Step 27.5
# Compare two supposedly different variations

for index in [0, 6]:

    item = ranked_test_variations[index]
    itinerary = item["itinerary"]

    print(f"\n{'=' * 60}")
    print(f"VARIATION {index + 1}")
    print(f"{'=' * 60}")

    for day in itinerary["days"]:

        print(f"\nDate: {day['date']}")
        print(f"City: {day['city']}")

        print("Activities:")

        for activity in day.get("activities", []):
            print(
                f"  {activity['name']} | "
                f"{activity.get('start')} → "
                f"{activity.get('end')} | "
                f"City: {activity.get('city')}"
            )

        print("Transfers:")

        for transfer in day.get("transfers", []):
            print(f"  {transfer}")

In [ ]:
# Step 27.6 Test

for index in [0, 6]:

    itinerary = ranked_test_variations[index]["itinerary"]

    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    score = calculate_final_itinerary_score(
        itinerary,
        travel_times=travel_times
    )

    print(f"\nVariation {index + 1}")
    print("Travel efficiency:", features["average_travel_efficiency"])
    print("Total travel time:", features["total_travel_time"])
    print("Final score:", score)

In [ ]:
# Step 28.1
# Recalculate scores for all itinerary variations

rescored_variations = []

for item in ranked_test_variations:

    itinerary = item["itinerary"]

    new_score = calculate_final_itinerary_score(
        itinerary,
        travel_times=travel_times
    )

    rescored_variations.append({
        "itinerary": itinerary,
        "score": new_score
    })

print("Total rescored variations:", len(rescored_variations))

for i, item in enumerate(rescored_variations, start=1):
    print(
        f"Variation {i}: "
        f"Score = {item['score']}"
    )

In [ ]:
# Step 28.2
# Check target diversity after rescoring

scores = [
    item["score"]
    for item in rescored_variations
]

print("Scores:")
print(scores)

print("\nUnique scores:")
print(sorted(set(scores)))

print(
    "\nNumber of unique scores:",
    len(set(scores))
)

In [ ]:
# Step 29
# Rebuild X and y using the corrected itinerary scores

X = []
y = []

for item in rescored_variations:

    itinerary = item["itinerary"]
    score = item["score"]

    features = extract_itinerary_features(
        itinerary,
        travel_times=travel_times
    )

    vector = create_feature_vector(features)

    X.append(vector)
    y.append(score)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX:")
print(X)

print("\ny:")
print(y)

print("\nUnique target scores:")
print(np.unique(y))

In [ ]:
# Step 31
# Inspect the complete canonical itinerary

for day in canonical_itinerary["days"]:

    print("\n" + "=" * 60)
    print(f"DATE: {day['date']}")
    print(f"CITY: {day['city']}")

    print("\nActivities:")

    for activity in day.get("activities", []):

        print(
            f"  {activity['name']} | "
            f"{activity.get('start')} → "
            f"{activity.get('end')} | "
            f"Duration: {activity.get('duration_minutes')} | "
            f"Type: {activity.get('duration_type')}"
        )

    print("\nTransfers:")

    for transfer in day.get("transfers", []):

        print(f"  {transfer}")

In [ ]:
# Step 32.1
# Inspect activity locations in the canonical itinerary

for day in canonical_itinerary["days"]:

    for activity in day.get("activities", []):

        print(
            f"{activity['name']}"
            f" | City: {activity.get('city')}"
            f" | Locations: {activity.get('locations')}"
        )

In [ ]:
# Step 33.1
# Inspect the travel-time matrix

print("Number of travel-time entries:", len(travel_times))

for key, value in travel_times.items():
    print(f"{key} -> {value} minutes")

In [ ]:
# Step 33.2
# Check which itinerary locations have travel-time data

activity_locations = set()

for day in canonical_itinerary["days"]:
    for activity in day.get("activities", []):
        for location in activity.get("locations", []):
            activity_locations.add(location)

print("Total unique itinerary locations:")
print(len(activity_locations))

print("\nLocations:")
for location in sorted(activity_locations):
    print("-", location)

In [ ]:
# Step 34.1
# Build the list of activities in the canonical itinerary

activities = []

for day in canonical_itinerary["days"]:

    for activity in day.get("activities", []):

        if activity["name"] != "Free Day":
            activities.append({
                "name": activity["name"],
                "city": activity.get("city"),
                "locations": activity.get("locations", [])
            })

print("Total activities:", len(activities))

for i, activity in enumerate(activities, start=1):
    print(
        f"{i}. {activity['name']} "
        f"| City: {activity['city']}"
    )

In [ ]:
# Step 34.2
# Generate possible activity-to-activity transitions

activity_transitions = []

for source in activities:

    for destination in activities:

        if source["name"] == destination["name"]:
            continue

        activity_transitions.append({
            "from": source["name"],
            "to": destination["name"],
            "from_city": source["city"],
            "to_city": destination["city"]
        })

print(
    "Possible activity transitions:",
    len(activity_transitions)
)

for transition in activity_transitions:
    print(
        f"{transition['from']} "
        f"({transition['from_city']})"
        f" → "
        f"{transition['to']} "
        f"({transition['to_city']})"
    )

In [ ]:
# Step 35.1
# Extract explicit inter-city transfers from the canonical itinerary

inter_city_transfers = {}

for day in canonical_itinerary["days"]:

    for transfer in day.get("transfers", []):

        if transfer.get("type") == "transfer":

            key = (
                transfer["from"],
                transfer["to"]
            )

            inter_city_transfers[key] = (
                transfer["duration_minutes"]
            )

print("Known inter-city transfers:")

for key, duration in inter_city_transfers.items():

    print(
        f"{key[0]} → {key[1]} "
        f"= {duration} minutes"
    )

In [ ]:
# Step 35.2
# Identify activity transitions that stay within the same city

intra_city_transitions = []

for source in activities:

    for destination in activities:

        if source["name"] == destination["name"]:
            continue

        if source["city"] == destination["city"]:

            intra_city_transitions.append(
                (
                    source["name"],
                    destination["name"],
                    source["city"]
                )
            )

print(
    "Intra-city transitions:",
    len(intra_city_transitions)
)

for transition in intra_city_transitions:

    print(
        f"{transition[0]} → "
        f"{transition[1]} "
        f"({transition[2]})"
    )

In [ ]:
# Step 35.3
# Create the real travel-time matrix structure

real_travel_times = {}

# ---------------------------------------
# Intra-city transitions
# ---------------------------------------

for source, destination, city in intra_city_transitions:

    real_travel_times[
        (source, destination)
    ] = None


# ---------------------------------------
# Inter-city transfers
# ---------------------------------------

for (source_city, destination_city), duration in inter_city_transfers.items():

    real_travel_times[
        (source_city, destination_city)
    ] = duration


print(
    "Total travel-time entries:",
    len(real_travel_times)
)

print("\nMatrix:")

for key, value in real_travel_times.items():

    print(
        f"{key[0]} → {key[1]} = {value}"
    )

In [ ]:
# Step 37.1
# Real activity-level travel matrix structure

real_activity_travel_times = {}

hong_kong_activities = [
    "Hong Kong Night Tour",
    "Hong Kong Disneyland",
    "Lantau Island Tour",
    "Ocean Park"
]

for source in hong_kong_activities:

    for destination in hong_kong_activities:

        if source == destination:
            continue

        real_activity_travel_times[
            (source, destination)
        ] = None


print(
    "Hong Kong activity transitions:",
    len(real_activity_travel_times)
)

for key, value in real_activity_travel_times.items():

    print(
        f"{key[0]} → {key[1]} = {value}"
    )

In [ ]:
# Step 37.2
# Demo travel-time matrix for Hong Kong activities
#
# These are estimated values for pipeline development.
# Replace with routing/API-derived values for production.

real_activity_travel_times = {

    # Hong Kong Night Tour
    ("Hong Kong Night Tour", "Hong Kong Disneyland"): 60,
    ("Hong Kong Night Tour", "Lantau Island Tour"): 45,
    ("Hong Kong Night Tour", "Ocean Park"): 35,

    # Hong Kong Disneyland
    ("Hong Kong Disneyland", "Hong Kong Night Tour"): 60,
    ("Hong Kong Disneyland", "Lantau Island Tour"): 25,
    ("Hong Kong Disneyland", "Ocean Park"): 45,

    # Lantau Island Tour
    ("Lantau Island Tour", "Hong Kong Night Tour"): 45,
    ("Lantau Island Tour", "Hong Kong Disneyland"): 25,
    ("Lantau Island Tour", "Ocean Park"): 50,

    # Ocean Park
    ("Ocean Park", "Hong Kong Night Tour"): 35,
    ("Ocean Park", "Hong Kong Disneyland"): 45,
    ("Ocean Park", "Lantau Island Tour"): 50
}

print(
    "Total travel-time entries:",
    len(real_activity_travel_times)
)

for key, value in real_activity_travel_times.items():
    print(
        f"{key[0]} → {key[1]} = {value} minutes"
    )

In [ ]:
# Step 37.3
# Validate travel-time matrix against activity transitions

expected_transitions = {
    (source, destination)
    for source, destination, city
    in intra_city_transitions
}

actual_transitions = set(
    real_activity_travel_times.keys()
)

missing = expected_transitions - actual_transitions
extra = actual_transitions - expected_transitions

print("Expected transitions:", len(expected_transitions))
print("Actual transitions:", len(actual_transitions))

print("\nMissing transitions:")
print(missing)

print("\nUnexpected transitions:")
print(extra)

if not missing and not extra:
    print("\nTravel matrix validation: PASSED")
else:
    print("\nTravel matrix validation: FAILED")

In [ ]:
# Step 38.1
# Extract Hong Kong activities from the canonical itinerary

hong_kong_days = []

for day in canonical_itinerary["days"]:

    if day["city"] != "Hong Kong":
        continue

    for activity in day.get("activities", []):

        if activity["name"] == "Free Day":
            continue

        hong_kong_days.append({
            "date": day["date"],
            "city": day["city"],
            "activity": activity
        })

print(
    "Hong Kong activities:",
    len(hong_kong_days)
)

for item in hong_kong_days:

    activity = item["activity"]

    print(
        item["date"],
        "→",
        activity["name"],
        "|",
        activity.get("start"),
        "→",
        activity.get("end")
    )

In [ ]:
# Step 38.2
# Generate all possible Hong Kong activity orders

from itertools import permutations

hk_activities = [
    item["activity"]
    for item in hong_kong_days
]

hong_kong_order_variations = [
    list(order)
    for order in permutations(hk_activities)
]

print(
    "Total possible Hong Kong orders:",
    len(hong_kong_order_variations)
)

for i, order in enumerate(
    hong_kong_order_variations,
    start=1
):

    print(
        f"Variation {i}:",
        [
            activity["name"]
            for activity in order
        ]
    )

In [ ]:
# Step 38.3
# Build candidate Hong Kong itineraries from activity permutations

def build_hong_kong_variation(
    canonical_itinerary,
    activity_order
):
    """
    Create a candidate itinerary by assigning
    a permutation of Hong Kong activities to
    the existing Hong Kong dates.

    Non-Hong-Kong days remain unchanged.
    """

    new_itinerary = []

    hong_kong_dates = [
        day
        for day in canonical_itinerary["days"]
        if day["city"] == "Hong Kong"
    ]

    hk_index = 0

    for day in canonical_itinerary["days"]:

        new_day = day.copy()

        # --------------------------------
        # Hong Kong day
        # --------------------------------

        if day["city"] == "Hong Kong":

            activity = activity_order[hk_index]

            new_day["activities"] = [
                activity.copy()
            ]

            hk_index += 1

        # --------------------------------
        # Other cities unchanged
        # --------------------------------

        new_itinerary.append(
            new_day
        )

    return new_itinerary


# Build all 24 candidate itineraries

candidate_hk_itineraries = []

for order in hong_kong_order_variations:

    candidate = build_hong_kong_variation(
        canonical_itinerary,
        order
    )

    candidate_hk_itineraries.append(
        candidate
    )


print(
    "Candidate itineraries:",
    len(candidate_hk_itineraries)
)

In [ ]:
# Step 38.4
# Inspect candidate itinerary ordering

for variation_number in [1, 2, 7, 24]:

    itinerary = candidate_hk_itineraries[
        variation_number - 1
    ]

    print(
        "\n" + "=" * 60
    )

    print(
        f"VARIATION {variation_number}"
    )

    print(
        "=" * 60
    )

    for day in itinerary:

        if day["city"] != "Hong Kong":
            continue

        names = [
            activity["name"]
            for activity
            in day.get("activities", [])
        ]

        print(
            day["date"],
            "→",
            names
        )

In [ ]:
# Step 39.1
# Validate all generated Hong Kong variations

valid_hk_variations = []
invalid_hk_variations = []

for i, itinerary_days in enumerate(
    candidate_hk_itineraries,
    start=1
):

    result = validate_itinerary_variation(
        itinerary_days
    )

    if result:
        valid_hk_variations.append(
            {
                "variation": i,
                "itinerary": itinerary_days
            }
        )
    else:
        invalid_hk_variations.append(
            {
                "variation": i,
                "itinerary": itinerary_days
            }
        )


print(
    "Total candidates:",
    len(candidate_hk_itineraries)
)

print(
    "Valid variations:",
    len(valid_hk_variations)
)

print(
    "Invalid variations:",
    len(invalid_hk_variations)
)

In [ ]:
# Step 40.1
# Extract features and calculate scores
# for all 24 valid Hong Kong variations

variation_results = []

for item in valid_hk_variations:

    variation_number = item["variation"]
    itinerary_days = item["itinerary"]

    # Convert list of days into canonical structure
    itinerary = {
        "schema_version": "1.0",
        "voucher_id": (
            f"hk_variation_{variation_number}"
        ),
        "days": itinerary_days
    }

    # Extract ML features
    features = extract_itinerary_features(
        itinerary,
        travel_times=real_activity_travel_times
    )

    # Calculate final quality score
    score = calculate_final_itinerary_score(
        itinerary
    )

    variation_results.append({
        "variation": variation_number,
        "itinerary": itinerary,
        "features": features,
        "score": score
    })


print(
    "Total ML samples:",
    len(variation_results)
)

In [ ]:
# Step 40.2
# Inspect extracted features and scores

print("Number of samples:", len(variation_results))

for item in variation_results[:3]:
    print("\nVariation:", item["variation"])
    print("Features:", item["features"])
    print("Score:", item["score"])

In [ ]:
for item in valid_hk_variations[:5]:
    print("\nVariation:", item["variation"])

    for day in item["itinerary"]:
        print(
            day.get("date"),
            "|",
            day.get("city"),
            "|",
            [
                activity.get("name")
                for activity in day.get("activities", [])
            ]
        )

In [ ]:
# Step 40.2A
# Inspect the itineraries stored inside variation_results

print("Total variations:", len(variation_results))

for item in variation_results[:5]:

    print("\n==============================")
    print("Variation:", item["variation"])
    print("Score:", item["score"])
    print("Features:", item["features"])

    for day in item["itinerary"]["days"]:

        print(
            day.get("date"),
            "|",
            day.get("city"),
            "|",
            [
                activity.get("name")
                for activity in day.get("activities", [])
            ]
        )